# Aarohan FMCG Analytics — Guided Notebook

This walkthrough uses the included synthetic data. Run the Python script from the repository root to rebuild full outputs. Use the notebook to inspect grain, data quality, metrics and summaries.


In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
DATA = ROOT / 'data'
OUT = ROOT / 'outputs'
orders = pd.read_csv(DATA / 'orders_raw.csv')
products = pd.read_csv(DATA / 'dim_products.csv')
print('Raw order-line rows:', len(orders))
print('Exact duplicate rows:', orders.duplicated().sum())
print('Distinct order IDs:', orders['order_id'].nunique())
display(orders.head())


## 1. Understand the grain
A row is an order line, not necessarily a complete order. `order_id` can repeat across different lines; `order_line_id` identifies a line. Exact duplicate rows should be inspected separately from repeated order IDs.


In [ ]:
analysis_orders = orders.drop_duplicates().copy()
print('Rows after exact duplicate removal:', len(analysis_orders))
print('Unique order lines:', analysis_orders['order_line_id'].nunique())
print('Rows with discount outside 0–1:', ((analysis_orders['discount_pct'] < 0) | (analysis_orders['discount_pct'] > 1)).sum())


## 2. Build transaction metrics
The project uses explicit definitions for gross sales, discount amount and net sales before returns. The return and profit treatment is an estimate because the source does not define an accounting policy.


In [ ]:
analysis_orders['gross_sales'] = analysis_orders['quantity'] * analysis_orders['unit_price']
analysis_orders['discount_amount'] = analysis_orders['gross_sales'] * analysis_orders['discount_pct']
analysis_orders['net_sales_pre_returns'] = analysis_orders['gross_sales'] - analysis_orders['discount_amount']
analysis_orders[['order_id','order_line_id','quantity','unit_price','discount_pct','gross_sales','discount_amount','net_sales_pre_returns']].head()


## 3. Rebuild outputs and reconcile
Run `python python/aarohan_analysis.py` from the repository root. Compare generated totals with `outputs/executive_summary.csv`, `outputs/yearly_summary.csv`, and the summaries for product, region and channel.
